# Data Pull — Kalshi Prediction Market Trades

This notebook pulls data from Kalshi's public REST API for the 2
markets used in this project and saves them to desktop as CSVs.

**Markets:**
- `KXNCAAF-26-IND` — Indiana wins the 2026 CFP National Championship (resolved YES)
- `KXFEDCHAIRNOM-29-KW` — Kevin Warsh nominated as Fed Chair (resolved YES)




## 1. Setup

In [17]:
import requests
import pandas as pd
import time
import os

BASE_URL = "https://api.elections.kalshi.com/trade-api/v2"

# Where the CSVs will be written
DATA_DIR = os.path.expanduser("~/Desktop")

## 2. Locating the target markets

The two events of
interest are the CFP championship (`KXNCAAF-26`) and the Fed Chair nomination
(`KXFEDCHAIRNOM-29`). The cells below confirm the event tickers and the winning submarket.

### 2a. Find the Fed Chair nomination market

In [18]:
#look up event
r = requests.get(f"{BASE_URL}/events/KXFEDCHAIRNOM-29")
print(f"Event: {r.json().get('event', {}).get('title')}\n")

#look up the Warsh sub-market
r = requests.get(f"{BASE_URL}/historical/markets/KXFEDCHAIRNOM-29-KW")
m = r.json().get("market", {})
print(f"Market:           {m.get('ticker')}")
print(f"Title:            {m.get('title')}")
print(f"Result:           {m.get('result')}")
print(f"Lifetime volume:  {float(m.get('volume_fp', 0)):,.0f} contracts")

Event: Who will Trump nominate as Fed Chair?

Market:           KXFEDCHAIRNOM-29-KW
Title:            Will Trump next nominate Kevin Warsh as Fed Chair?
Result:           yes
Lifetime volume:  38,983,491 contracts


### 2b. Find the CFP championship market

In [19]:
#look up event
r = requests.get(f"{BASE_URL}/events/KXNCAAF-26")
print(f"Event: {r.json().get('event', {}).get('title')}\n")

#look up Indiana sub-market
r = requests.get(f"{BASE_URL}/historical/markets/KXNCAAF-26-IND")
m = r.json().get("market", {})
print(f"Market:           {m.get('ticker')}")
print(f"Title:            {m.get('title')}")
print(f"Result:           {m.get('result')}")
print(f"Lifetime volume:  {float(m.get('volume_fp', 0)):,.0f} contracts")
   

    

Event: College Football Championship: Miami vs Indiana

Market:           KXNCAAF-26-IND
Title:            Will the Indiana win the College Football Playoff National Championship?
Result:           yes
Lifetime volume:  33,574,443 contracts


## 3. Trade-pull function

Kalshi's `/historical/trades` endpoint returns settled-market trades in pages of
up to 1,000, navigated with a `cursor`. The function below pages through all
trades for a given ticker and returns them as a DataFrame.

In [20]:
def get_historical_trades(ticker, max_pages=None, verbose=True):
    all_trades = []
    cursor = None
    page = 0

    while True:
        params = {"ticker": ticker, "limit": 1000}
        if cursor:
            params["cursor"] = cursor

        r = requests.get(f"{BASE_URL}/historical/trades", params=params)
        if r.status_code != 200:
            print(f"Error {r.status_code}: {r.text[:300]}")
            break

        data = r.json()
        trades = data.get("trades", [])
        all_trades.extend(trades)
        page += 1

        cursor = data.get("cursor")
        if not cursor or len(trades) == 0:
            break
        if max_pages and page >= max_pages:
            break

    print(f"  done: {len(all_trades):,} total trades for {ticker}")
    return pd.DataFrame(all_trades)


# Columns kept for analysis (the API returns a few extra book-side fields we don't use)
KEEP_COLS = ["trade_id", "ticker", "created_time", "count_fp",
             "yes_price_dollars", "no_price_dollars", "taker_side"]

## 4. Pull each market and save to CSV

Each market is pulled, trimmed to the analysis columns, and saved to desktop.

In [21]:
markets_to_pull = {
    "KXNCAAF-26-IND":      "indiana_championship_trades.csv",
    "KXFEDCHAIRNOM-29-KW": "warsh_fed_chair_trades.csv",
}

for ticker, filename in markets_to_pull.items():
    print(f"Pulling {ticker} ...")
    df = get_historical_trades(ticker)
    df = df[[c for c in KEEP_COLS if c in df.columns]]
    out_path = os.path.join(DATA_DIR, filename)
    df.to_csv(out_path, index=False)
    print(f"  saved {len(df):,} rows -> {out_path}\n")

Pulling KXNCAAF-26-IND ...
  done: 134,549 total trades for KXNCAAF-26-IND
  saved 134,549 rows -> /Users/josephsemelroth/Desktop/indiana_championship_trades.csv

Pulling KXFEDCHAIRNOM-29-KW ...
  done: 71,969 total trades for KXFEDCHAIRNOM-29-KW
  saved 71,969 rows -> /Users/josephsemelroth/Desktop/warsh_fed_chair_trades.csv

